# 🇰🇪 Job Hunter KE — Control Notebook

Unified notebook for **jobs** and **scholarships**. Uses free-tier AI (Groq / OpenRouter / Ollama).

```
JOBS:         Scrape → Generate docs → Review → Approve → Export .md
SCHOLARSHIPS: Scrape → Generate SOP/Research Proposal → Review → Approve → Export .md
```

**Sections:**
- Cells 1–2:   Setup & profile
- Cells 3–8:   Jobs workflow
- Cells 9–15:  Scholarships workflow
- Cell  16:    Combined stats

In [ ]:
# ── CELL 1: Setup ─────────────────────────────────────────────────────────
import os, json, uuid, datetime, re
from collections import Counter
import pandas as pd
from IPython.display import display, Markdown, HTML
from dotenv import load_dotenv

load_dotenv()

from scrapers             import scrape_all
from scholarship_scrapers import scrape_scholarships, SCHOLARSHIP_SOURCES
from ai_engine import (
    generate_application_package,
    generate_scholarship_package,
    polish_markdown,
    MODEL,
)
from db import JobDB

db = JobDB('jobs.db')
print(f'✅ Setup complete')
print(f'🤖 Model: {MODEL}')
print(f'📊 DB:    {db.pipeline_summary()}')

In [ ]:
# ── CELL 2: Load profile ──────────────────────────────────────────────────
with open('my_profile.json') as f:
    PROFILE = json.load(f)

print(f"👤 {PROFILE['name']}")
print(f"🎯 Roles:   {', '.join(PROFILE['target_roles'])}")
print(f"🏢 Sectors: {', '.join(PROFILE['target_sectors'])}")

---
## 💼 JOBS WORKFLOW
---

In [ ]:
# ── CELL 3: Scrape jobs ────────────────────────────────────────────────────
JOB_KEYWORDS = PROFILE['target_roles']
JOB_SOURCES  = [
    'neaims', 'gaa',                          # 🏛️ Kenya Government
    'brightermonday', 'myjobmag', 'fuzu',      # 🇰🇪 Kenya Private
    'jobwebkenya', 'careersinkenya',
    'ngojobskenya',                            # 🌍 NGO/INGO
    'linkedin', 'reddit',                      # 🌐 Global
]

print('🔍 Scraping jobs...')
jobs = scrape_all(JOB_KEYWORDS, JOB_SOURCES)
saved = 0
for job in jobs:
    if not db.exists(job.get('url',''), 'job'):
        job.update({'id': str(uuid.uuid4()), 'status': 'pending',
                    'record_type': 'job', 'scraped_at': datetime.datetime.utcnow().isoformat()})
        if db.save(job): saved += 1

print(f'📥 Scraped: {len(jobs)}  |  💾 New: {saved}  |  ♻️  Dupes skipped: {len(jobs)-saved}')

In [ ]:
# ── CELL 4: View pending jobs ─────────────────────────────────────────────
pending_jobs = db.get_by_status('pending', 'job')
if not pending_jobs:
    print('No pending jobs.')
else:
    df = pd.DataFrame(pending_jobs)[['title','company','location','source','sector','deadline']]
    pd.set_option('display.max_colwidth', 50)
    display(df)

In [ ]:
# ── CELL 5: Generate job application docs ────────────────────────────────
MAX_JOBS = 10   # Groq free: ~30 req/min → 10 jobs ≈ 1 min

pending_jobs = db.get_by_status('pending', 'job')[:MAX_JOBS]
print(f'🤖 Generating docs for {len(pending_jobs)} jobs...')

for i, job in enumerate(pending_jobs, 1):
    print(f'[{i}/{len(pending_jobs)}] {job["title"]} @ {job["company"]}')
    try:
        pkg = generate_application_package(job, PROFILE)
        db.update(job['id'], {
            'cover_letter': pkg['cover_letter_md'],
            'tailored_resume': pkg['tailored_resume'],
            'full_doc_md': pkg['full_doc_md'],
            'match_score': pkg['match_score'],
            'status': 'ready_for_review',
        }, 'job')
        print(f'  ✅ Match: {pkg["match_score"]}%')
    except Exception as e:
        print(f'  ❌ {e}')

In [ ]:
# ── CELL 6: Preview a job application doc ────────────────────────────────
ready_jobs = db.get_by_status('ready_for_review', 'job')
ready_jobs.sort(key=lambda j: j.get('match_score',0), reverse=True)

IDX = 0   # change to preview a different job
job = ready_jobs[IDX]

print(f"Role:    {job['title']}  |  Company: {job['company']}")
print(f"Source:  {job['source']}  |  Sector: {job.get('sector','')}  |  Match: {job.get('match_score','?')}%")
print(f"URL:     {job['url']}")
print('─'*60)
display(Markdown(job.get('full_doc_md') or job.get('cover_letter','')))

In [ ]:
# ── CELL 7: Approve / reject job ─────────────────────────────────────────
# Edit the cover letter or full doc below before approving:
edited_doc = job.get('full_doc_md','')   # modify this string as needed

ACTION = 'approve'   # 'approve' or 'reject'

if ACTION == 'approve':
    from ai_engine import assemble_application_doc
    db.update(job['id'], {
        'status': 'approved',
        'full_doc_md': edited_doc,
        'approved_at': datetime.datetime.utcnow().isoformat()
    }, 'job')
    print(f'✅ Approved: {job["title"]} @ {job["company"]}')
else:
    db.update(job['id'], {'status': 'rejected'}, 'job')
    print(f'❌ Rejected: {job["title"]}')

In [ ]:
# ── CELL 8: Export approved jobs to .md files ─────────────────────────────
os.makedirs('output/jobs', exist_ok=True)
approved_jobs = db.get_by_status('approved', 'job')
print(f'Exporting {len(approved_jobs)} approved job applications...')

for job in approved_jobs:
    safe = lambda s: re.sub(r'[^\w\-]','_',s or 'unknown')[:28]
    fname = f"output/jobs/{safe(job.get('company'))}__{safe(job.get('title'))}.md"
    with open(fname,'w',encoding='utf-8') as f:
        f.write(job.get('full_doc_md') or job.get('cover_letter',''))
    print(f'  💾 {fname}')

print('\n💡 Convert to PDF: pandoc file.md -o file.pdf --pdf-engine=wkhtmltopdf')

---
## 🎓 SCHOLARSHIPS WORKFLOW
---

In [ ]:
# ── CELL 9: Browse available scholarship sources ──────────────────────────
print(f'📚 {len(SCHOLARSHIP_SOURCES)} scholarship sources registered:\n')
for key, (country, region, funding) in sorted(SCHOLARSHIP_SOURCES.items(), key=lambda x: x[1][1]):
    icon = {'europe':'🇪🇺','asia':'🌏','americas':'🌎','africa':'🌍','global':'🌐'}.get(region,'🔵')
    ft   = '🟢 Full' if funding == 'fully_funded' else ('🟡 Partial' if funding == 'partially_funded' else '⚪ Mixed')
    print(f'  {icon} {key:<28} {country:<15} {ft}')

In [ ]:
# ── CELL 10: Scrape scholarships ──────────────────────────────────────────
# Keywords: field of study / subject area (NOT job titles)
SCHOL_KEYWORDS = [
    'data science',
    'public health',
    'development studies',
    'statistics',
    'computer science',
]

# To scrape all sources (slower, ~20-40 min):
#   schol_sources = None
# To scrape specific sources:
schol_sources = [
    # 🌐 High-value fully-funded
    'chevening', 'commonwealth',
    'daad', 'heinrich_boell',
    'fulbright', 'hubert_humphrey',
    'mext', 'jica',
    'australia_awards',
    'nuffic_okp', 'orange_knowledge',
    'swedish_institute',
    'vlir_uos',
    'swiss_eskas',
    'turkiye_burslari',
    'csc_china',
    'gks_korea',
    'mastercard_scholars',
    'aga_khan',
    'rotary_peace',
    # 🌍 Aggregators
    'scholars4dev',
    'scholarshipdb',
    'opportunitiesforafricans',
    'afterschoolafrica',
    'scholarshippositions',
]

# Optional filters:
REGION_FILTER  = None          # 'europe' / 'asia' / 'americas' / None
FUNDING_FILTER = 'fully_funded'  # 'fully_funded' / 'partially_funded' / None

print('🔍 Scraping scholarships...')
scholarships = scrape_scholarships(
    keywords       = SCHOL_KEYWORDS,
    sources        = schol_sources,
    region_filter  = REGION_FILTER,
    funding_filter = FUNDING_FILTER,
)

saved = 0
for s in scholarships:
    if not db.scholarship_exists(s.get('url','')):
        s.update({'id': str(uuid.uuid4()), 'status': 'pending',
                  'scraped_at': datetime.datetime.utcnow().isoformat()})
        if db.save_scholarship(s): saved += 1

print(f'📥 Scraped: {len(scholarships)}  |  💾 New: {saved}')

In [ ]:
# ── CELL 11: View pending scholarships ───────────────────────────────────
pending_schol = db.get_scholarships(status='pending')
if not pending_schol:
    print('No pending scholarships. Run Cell 10 to scrape.')
else:
    df = pd.DataFrame(pending_schol)[[
        'title','funder_country','level','funding_type','deadline','source'
    ]]
    pd.set_option('display.max_colwidth', 52)
    display(df)
    
    print(f'\n🟢 Fully funded:    {sum(1 for s in pending_schol if s.get("funding_type")=="fully_funded")}')
    print(f'🟡 Partially funded: {sum(1 for s in pending_schol if s.get("funding_type")=="partially_funded")}')
    countries = Counter(s.get('funder_country','?') for s in pending_schol)
    print('\n📌 By country:', dict(countries.most_common(10)))

In [ ]:
# ── CELL 12: Generate scholarship docs (SOP + CV insights) ───────────────
# Each scholarship = 2-3 AI calls. Start with 5 for testing.
MAX_SCHOL = 5

# Optional: only process specific funding type or region
pending_schol = db.get_scholarships(status='pending', funding_type='fully_funded')[:MAX_SCHOL]
print(f'🤖 Generating scholarship packages for {len(pending_schol)} scholarships...')
print(f'   Model: {MODEL}\n')

for i, s in enumerate(pending_schol, 1):
    print(f'[{i}/{len(pending_schol)}] {s["title"]}')
    print(f'   Funder: {s.get("funder_country","?")} · Level: {s.get("level","?")} · {s.get("funding_type","?")} ')
    try:
        pkg = generate_scholarship_package(s, PROFILE)
        db.update_scholarship(s['id'], {
            'motivation_letter': pkg['motivation_letter'],
            'research_proposal': pkg['research_proposal'],
            'tailored_resume':   pkg['tailored_resume'],
            'full_doc_md':       pkg['full_doc_md'],
            'match_score':       pkg['match_score'],
            'status':            'ready_for_review',
        })
        print(f'   ✅ Match: {pkg["match_score"]}%')
    except Exception as e:
        print(f'   ❌ {e}')

print('\n✅ Done. Proceed to Cell 13 to review.')

In [ ]:
# ── CELL 13: Preview a scholarship application package ────────────────────
ready_schol = db.get_scholarships(status='ready_for_review')
ready_schol.sort(key=lambda s: s.get('match_score',0), reverse=True)

if not ready_schol:
    print('No scholarships ready. Run Cell 12 first.')
else:
    IDX = 0   # change index to preview different scholarships
    s = ready_schol[IDX]

    print(f"Scholarship: {s['title']}")
    print(f"Funder:      {s.get('funder','?')} ({s.get('funder_country','?')})")
    print(f"Level:       {s.get('level','?')}  |  Funding: {s.get('funding_type','?')}")
    print(f"Coverage:    {s.get('coverage','?')}")
    print(f"Deadline:    {s.get('deadline','?')}")
    print(f"Match:       {s.get('match_score','?')}%")
    print(f"URL:         {s['url']}")
    print('─'*65)

    # Render the full application package as Markdown
    display(Markdown(s.get('full_doc_md') or s.get('motivation_letter','')))

In [ ]:
# ── CELL 14: Approve / reject scholarship ────────────────────────────────
# s is set from Cell 13. Edit full_doc_md string before approving.
edited_doc = s.get('full_doc_md', '')
# e.g.: edited_doc = edited_doc.replace('old phrase', 'better phrase')

ACTION = 'approve'   # 'approve' or 'reject'

if ACTION == 'approve':
    db.update_scholarship(s['id'], {
        'status': 'approved',
        'full_doc_md': edited_doc,
        'approved_at': datetime.datetime.utcnow().isoformat()
    })
    print(f'✅ Approved: {s["title"]}')
else:
    db.update_scholarship(s['id'], {'status': 'rejected'})
    print(f'❌ Rejected: {s["title"]}')

In [ ]:
# ── CELL 15: Export approved scholarships to .md files ───────────────────
os.makedirs('output/scholarships', exist_ok=True)
approved_schol = db.get_scholarships(status='approved')
print(f'Exporting {len(approved_schol)} approved scholarship packages...')

safe = lambda s: re.sub(r'[^\w\-]','_', s or 'unknown')[:35]

for s in approved_schol:
    fname = f"output/scholarships/{safe(s.get('funder_country'))}__{safe(s.get('title'))}.md"
    with open(fname,'w',encoding='utf-8') as f:
        f.write(s.get('full_doc_md') or s.get('motivation_letter',''))
    print(f'  💾 {fname}')

print('\n💡 Convert to PDF:')
print('   pandoc file.md -o file.pdf --pdf-engine=wkhtmltopdf')
print('   or: grip file.md  (markdown preview in browser)')

In [ ]:
# ── CELL 16: Combined pipeline stats dashboard ───────────────────────────
s = db.stats()
p = db.pipeline_summary()

print('='*55)
print('  JOB HUNTER KE — PIPELINE DASHBOARD')
print('='*55)
print('\n💼 JOBS')
print('─'*30)
icons = {'pending':'⏳','ready_for_review':'🔍','approved':'✅','rejected':'❌','dispatched':'🚀'}
for status, count in s.get('jobs',{}).items():
    print(f"  {icons.get(status,'•')} {status:<22}: {count}")

print('\n🎓 SCHOLARSHIPS')
print('─'*30)
for status, count in s.get('scholarships',{}).items():
    print(f"  {icons.get(status,'•')} {status:<22}: {count}")
    
print('\n📊 SCHOLARSHIPS BY FUNDING')
print('─'*30)
for ft, count in s.get('scholarships_by_funding',{}).items():
    icon = '🟢' if ft == 'fully_funded' else ('🟡' if ft == 'partially_funded' else '⚪')
    print(f"  {icon} {ft:<22}: {count}")

print('\n🌍 SCHOLARSHIPS BY REGION')
print('─'*30)
for region, count in sorted(s.get('scholarships_by_region',{}).items(), key=lambda x: -x[1]):
    ricon = {'europe':'🇪🇺','asia':'🌏','americas':'🌎','africa':'🌍','global':'🌐'}.get(region,'🔵')
    print(f"  {ricon} {region:<22}: {count}")
print('='*55)